<a href="https://colab.research.google.com/github/ChaMooKwan/jwt-storage-security-analysis/blob/main/Automative_Searching_for_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 키 발급 및 환경설정

In [ ]:
# 1. github settings -> Developer Settings
# 2. classic 토큰 발급 후 복사 (권한체크 x)
# 3. 메모장 열고 입력: KEY = '(복사한 키)'
# 4. '.env'로 저장
# 5. RAM에 업로드 후 아래 코드 실행
# 주의: code_keyword와 repo_query를 변경하여 검색 옵션 세팅 필요

# 자동화 검색 코드

In [ ]:
# 1 github dev setting에서 토큰 발급후, .env파일생성 KEY ='(발급한토큰)'
import os
from dotenv import load_dotenv

# .env 파일을 읽어 환경 변수로 설정
load_dotenv()

# os.get env를 통해 사용
GITHUB_TOKEN = os.getenv("KEY")

In [ ]:
import requests
import time
import sys
import os
from dotenv import load_dotenv

# ==========================================
# 1. 환경 설정 및 API 헤더 세팅
# ==========================================
load_dotenv()
GITHUB_TOKEN = os.getenv("KEY")

if not GITHUB_TOKEN or GITHUB_TOKEN == "YOUR_GITHUB_TOKEN_HERE":
    print("[!] .env 파일에 KEY(GitHub Token)가 정상적으로 설정되지 않았습니다.")
    sys.exit(1)

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# 특정 도메인(KEYWORDS) 필터링을 제거하고, 단일 검색 쿼리를 리스트 형태로 구성합니다.
QUERIES = [
    'jwt stars:>50'
]

# ==========================================
# 2. 기능 함수 정의
# ==========================================
def search_repositories(query, max_results_per_query=100):
    """특정 쿼리로 저장소 목록을 수집 (안전한 API 요청 적용)"""
    repos = []
    page = 1

    while len(repos) < max_results_per_query:
        url = "https://api.github.com/search/repositories"

        params = {
            'q': query,
            'per_page': 100,
            'page': page
        }

        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()
                items = data.get('items', [])

                if data.get('incomplete_results'):
                    print("      [!] GitHub 서버 타임아웃으로 부분 결과만 수신됨")

                if not items:
                    break

                repos.extend([item['full_name'] for item in items])
                page += 1
                time.sleep(2)

            elif response.status_code == 422:
                print(f"      [!] 쿼리 문법 오류(422) 발생. 건너뜁니다.")
                break
            elif response.status_code in [403, 429]:
                print("      [!] 저장소 검색 API 제한. 30초 대기...")
                time.sleep(30)
            else:
                break

        except requests.exceptions.RequestException as e:
            print(f"      [!] 네트워크 오류: {e}")
            break

    return repos[:max_results_per_query]

def has_docker_compose(repo_full_name):
    """저장소 내 docker-compose 파일 존재 여부 확인"""
    target_files = ['docker-compose.yml', 'docker-compose.yaml']
    for file_name in target_files:
        url = f"https://api.github.com/repos/{repo_full_name}/contents/{file_name}"
        response = requests.get(url, headers=HEADERS)

        if response.status_code == 200:
            return True
        elif response.status_code == 403 and 'rate limit' in response.text.lower():
            print("\n[!] File API Rate Limit 초과. 60초 대기...")
            time.sleep(60)
            return has_docker_compose(repo_full_name)
    return False

def has_jwt_sign_code(repo_full_name):
    """저장소 내 jwt.sign 코드 사용 여부 확인"""
    query = f"jwt.sign repo:{repo_full_name}"
    url = f"https://api.github.com/search/code?q={query}"

    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        items = response.json().get('items', [])
        return len(items) > 0
    elif response.status_code == 403:
        print(f"\n[!] Code Search API 한도 초과. 65초 대기 후 재시도합니다...")
        time.sleep(65)
        return has_jwt_sign_code(repo_full_name)
    return False

# ==========================================
# 3. 메인 자동화 파이프라인
# ==========================================
def main():
    print("=" * 60)
    print("🚀 [Step 1] 타겟 후보군 수집 시작")
    print("=" * 60)

    all_candidate_repos = set()


    for i, query in enumerate(QUERIES, 1):
        print(f"[*] 쿼리 {i}/{len(QUERIES)} 수집 중: {query}")
        repos = search_repositories(query, max_results_per_query=1000)
        all_candidate_repos.update(repos)
        print(f"    -> {len(repos)}개 수집 완료 (누적: {len(all_candidate_repos)}개)")
        time.sleep(2)

    candidate_list = list(all_candidate_repos)
    print(f"\n[*] 1차 후보군 총 {len(candidate_list)}개 저장소 수집 완료!\n")

    print("=" * 60)
    print("🔍 [Step 2 & 3] Docker 파일 검사 -> JWT 코드 검사 시작")
    print("=" * 60)

    final_targets = []
    output_file = "final_targets.txt"

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("=== 추출된 타겟 리스트 ===\n")

    for i, repo_name in enumerate(candidate_list, 1):
        sys.stdout.write(f"[-] 스캔 중 ({i}/{len(candidate_list)}): {repo_name} ... ")
        sys.stdout.flush()

        has_docker = has_docker_compose(repo_name)
        time.sleep(1)

        if not has_docker:
            print("[패스: Docker 파일 없음]")
            continue

        is_jwt_used = has_jwt_sign_code(repo_name)
        time.sleep(2.5)

        if is_jwt_used:
            print("[\033[92m최종 타겟 발견!\033[0m (Docker O, JWT O)]")
            target_url = f"https://github.com/{repo_name}"
            final_targets.append(target_url)

            with open(output_file, 'a', encoding='utf-8') as f:
                f.write(target_url + "\n")
        else:
            print("[패스: jwt.sign 미발견]")

    print("\n" + "=" * 60)
    print(f"🎉 모든 자동화 스캔 완료!")
    print(f"✅ 총 {len(final_targets)}개의 최종 시스템을 찾았습니다.")
    print(f"📂 결과는 '{output_file}'에 저장되었습니다.")
    print("=" * 60)

if __name__ == "__main__":
    main()

🚀 [Step 1] 타겟 후보군 수집 시작
[*] 쿼리 1/1 수집 중: jwt stars:>50
    -> 1000개 수집 완료 (누적: 1000개)

[*] 1차 후보군 총 1000개 저장소 수집 완료!

🔍 [Step 2 & 3] Docker 파일 검사 -> JWT 코드 검사 시작
[-] 스캔 중 (1/1000): burakorkmez/twitter-clone ... [패스: Docker 파일 없음]
[-] 스캔 중 (2/1000): zlt2000/microservices-platform ... [패스: Docker 파일 없음]
[-] 스캔 중 (3/1000): bradtraversy/node_jwt_example ... [패스: Docker 파일 없음]
[-] 스캔 중 (4/1000): maxgalbu/adonis5-jwt ... [패스: Docker 파일 없음]
[-] 스캔 중 (5/1000): AndrewJBateman/pern-stack-auth ... [패스: Docker 파일 없음]
[-] 스캔 중 (6/1000): hyperf-ext/jwt ... [패스: Docker 파일 없음]
[-] 스캔 중 (7/1000): bezkoder/vue-vuex-jwt-auth ... [패스: Docker 파일 없음]
[-] 스캔 중 (8/1000): PuZhiweizuishuai/SpringSecurity-JWT-Vue-Deom ... [패스: Docker 파일 없음]
[-] 스캔 중 (9/1000): codeforgeek/node-refresh-token ... [패스: Docker 파일 없음]
[-] 스캔 중 (10/1000): potatosalad/erlang-jose ... [패스: Docker 파일 없음]
[-] 스캔 중 (11/1000): dariusztytko/jwt-key-id-injector ... [패스: Docker 파일 없음]
[-] 스캔 중 (12/1000): ballerina-platform/module-ballerina-jwt 